# Filter EGMS data sets spatially

In [1]:
from pathlib import Path
import pandas as pd

## Filter spatially

In [5]:
# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

DATA_DIR = Path("/home/lukas/projects/ADUCAT/data/EGMS/extracted")

# ============================================================
# Input settings
# ============================================================

ORBIT = "022"
PRODUCT_START = "2020"
PRODUCT_END = "2024"

# ============================================================
# Find input files
# ============================================================

PRODUCT_RANGE = f"{PRODUCT_START}_{PRODUCT_END}"

csv_files = sorted(
    f for f in DATA_DIR.glob("*.csv")
    if (
        f"_{ORBIT}_" in f.name
        and f"_{PRODUCT_RANGE}_" in f.name
    )
)

print(
    f"Found {len(csv_files)} CSV files "
    f"for orbit {ORBIT}, "
    f"product period {PRODUCT_RANGE}"
)

for f in csv_files:
    print(f"  {f.name}")

OUTPUT_DIR = DATA_DIR.parent / "vienna"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Found 4 CSV files for orbit 022, product period 2020_2024
  EGMS_L2a_022_0786_IW1_VV_2020_2024_1.csv
  EGMS_L2a_022_0787_IW1_VV_2020_2024_1.csv
  EGMS_L2a_022_0788_IW1_VV_2020_2024_1.csv
  EGMS_L2a_022_0789_IW1_VV_2020_2024_1.csv


In [3]:
# ------------------------------------------------------------------
# Vienna bounding box
# ------------------------------------------------------------------
MIN_LON = 16.179211657826805
MAX_LON = 16.58969080849706
MIN_LAT = 48.11629276622462
MAX_LAT = 48.32510541429218

In [6]:
# ------------------------------------------------------------------
# Processing
# ------------------------------------------------------------------
CHUNK_SIZE = 100_000

In [7]:
for csv_file in csv_files:

    output_file = OUTPUT_DIR / csv_file.name

    print(f"\nProcessing: {csv_file.name}")

    first_chunk = True
    n_input = 0
    n_output = 0

    for chunk in pd.read_csv(csv_file, chunksize=CHUNK_SIZE):

        n_input += len(chunk)

        # Bounding-box filter
        mask = (
            (chunk["longitude"] >= MIN_LON) &
            (chunk["longitude"] <= MAX_LON) &
            (chunk["latitude"] >= MIN_LAT) &
            (chunk["latitude"] <= MAX_LAT)
        )

        filtered = chunk.loc[mask]

        n_output += len(filtered)

        # Write directly to disk
        if len(filtered) > 0:
            filtered.to_csv(
                output_file,
                mode="w" if first_chunk else "a",
                header=first_chunk,
                index=False
            )
            first_chunk = False

    print(f"  Input:  {n_input:,} points")
    print(f"  Vienna: {n_output:,} points")
    print(f"  Saved:  {output_file}")


Processing: EGMS_L2a_022_0786_IW1_VV_2020_2024_1.csv
  Input:  320,601 points
  Vienna: 0 points
  Saved:  /home/lukas/projects/ADUCAT/data/EGMS/vienna/EGMS_L2a_022_0786_IW1_VV_2020_2024_1.csv

Processing: EGMS_L2a_022_0787_IW1_VV_2020_2024_1.csv
  Input:  960,964 points
  Vienna: 513,591 points
  Saved:  /home/lukas/projects/ADUCAT/data/EGMS/vienna/EGMS_L2a_022_0787_IW1_VV_2020_2024_1.csv

Processing: EGMS_L2a_022_0788_IW1_VV_2020_2024_1.csv
  Input:  1,255,328 points
  Vienna: 823,196 points
  Saved:  /home/lukas/projects/ADUCAT/data/EGMS/vienna/EGMS_L2a_022_0788_IW1_VV_2020_2024_1.csv

Processing: EGMS_L2a_022_0789_IW1_VV_2020_2024_1.csv
  Input:  453,900 points
  Vienna: 0 points
  Saved:  /home/lukas/projects/ADUCAT/data/EGMS/vienna/EGMS_L2a_022_0789_IW1_VV_2020_2024_1.csv
